<a href="https://colab.research.google.com/github/pingbutwin/Web-Camera-with-face-recognition/blob/main/webCamServer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import cv2
import numpy as np
import uvicorn
import asyncio
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import StreamingResponse, HTMLResponse
!pip install ultralytics
from ultralytics import YOLO
import nest_asyncio
!pip install pyngrok
!pip install python-dotenv
from pyngrok import ngrok
import os
from dotenv import load_dotenv
from huggingface_hub import login
from huggingface_hub import hf_hub_download

load_dotenv()
nest_asyncio.apply()

In [3]:
login(os.getenv('HF_TOKEN'))
ngrok.set_auth_token(os.getenv('NGROK_AUTH_TOKEN'))

public_url = ngrok.connect(5050, proto='http')
print(f"Public URL: {public_url.public_url.replace('http', 'ws')}/ws/upload")

app = FastAPI(title='Yolo Webcam')

model_path = hf_hub_download(repo_id="arnabdhar/YOLOv8-Face-Detection", filename="model.pt")
model = YOLO(model_path)
last_processed_frame=None

log_label='[SERVER]:'
@app.websocket('/ws/upload')
async def websocket_endpoint(websocket: WebSocket):
    global last_processed_frame
    await websocket.accept()
    print(f'{log_label} websocket connected')

    try:
        while True:
            bytes_data = await websocket.receive_bytes()
            nparr = np.frombuffer(bytes_data, np.uint8)
            frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

            if frame is not None:
                results = model.predict(frame, imgsz=320, conf=0.25, verbose=False)
                annotated_frame = results[0].plot()

                ret, buffer = cv2.imencode('.jpg', annotated_frame)
                if not ret:
                  print(f'{log_label} WARNING: something went wrong with a frame')
                  continue
                last_processed_frame = buffer.tobytes()
    except WebSocketDisconnect:
      print(f'{log_label} ERROR: camera has disconnected')
    except Exception as e:
      print(f'{log_label} ERROR: {e}')

def generate_frames():
  global last_processed_frame
  while True:
    if last_processed_frame is not None:
      yield (b'--frame\r\n'
                   b'Content-Type: image/jpeg\r\n\r\n' + last_processed_frame + b'\r\n')

@app.get('/video_feed')
async def video_feed():
  return StreamingResponse(
      generate_frames(), media_type="multipart/x-mixed-replace; boundary=frame")

@app.get('/', response_class=HTMLResponse)
async def index():
  return """
    <html>
        <body style="background:#111; color:white; text-align:center;">
            <h1>YOLO Stream from Remote Client</h1>
            <img src="/video_feed" style="width:80%; border:2px solid #555;">
        </body>
    </html>
    """

async def run_server():
  config = uvicorn.Config(app, host='0.0.0.0', port=5050, loop='asyncio')
  server = uvicorn.Server(config)
  await server.serve()

def main():
  loop = asyncio.get_event_loop()
  server_task = loop.create_task(run_server())
  try:
    loop.run_forever()
    print(f'{log_label} Starting is running, press stop to shut down')
  except KeyboardInterrupt:
    print(f'{log_label} Shutting down')
  finally:
    server_task.cancel()
    ngrok.disconnect(public_url.public_url)
    ngrok.kill()
    loop.stop()

    print(f'{log_label} Everything has been shut down successfully')

if __name__ == '__main__':
  main()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Public URL: wss://brothy-lizeth-untactual.ngrok-free.dev/wss/upload


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
INFO:     Started server process [1613]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:5050 (Press CTRL+C to quit)
INFO:     2a02:a310:82c3:1380:803f:c2b:9d1c:39cb:0 - "WebSocket /wss/upload" 403
INFO:     connection rejected (403 Forbidden)
INFO:     connection closed
INFO:     2a02:a310:82c3:1380:803f:c2b:9d1c:39cb:0 - "WebSocket /ws/upload" [accepted]
INFO:     connection open


[SERVER]: websocket connected
INFO:     2a02:a310:82c3:1380:803f:c2b:9d1c:39cb:0 - "GET / HTTP/1.1" 200 OK
INFO:     2a02:a310:82c3:1380:803f:c2b:9d1c:39cb:0 - "GET /video_feed HTTP/1.1" 200 OK
INFO:     2a02:a310:82c3:1380:803f:c2b:9d1c:39cb:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found


INFO:     connection closed


[SERVER]: ERROR: camera has disconnected


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1613]


[SERVER]: Shutting down
[SERVER]: Everything has been shut down successfully
